In [2]:
import json
from collections import Counter
from openai import OpenAI
from tqdm import tqdm
import re

In [3]:
with open("logs/crawled_cat_candidates.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

In [4]:
data[:3]

[{'video_id': '3URtTIdnXIk',
  'video_title': 'These CATS are too FUNNY! 🤣 | New Cat Videos 2025',
  'score': 2,
  'source': 'search'},
 {'video_id': 'CUeHOro99rQ',
  'video_title': '🔴 24/7 LIVE: Cat TV 😺 Cute Birds Chipmunks and Squirrels 4K Cat Games',
  'score': 1,
  'source': 'search'},
 {'video_id': 'RefIZ5PeiTs',
  'video_title': 'Mama Cat Saves Kitten Falls Into the Swimming Pool Dry Fur, Cook and Lulling Kitten to Sleep 😿❤️',
  'score': 2,
  'source': 'search'}]

In [5]:
len(data)

12390

In [6]:
common_categories = Counter([x.get("categories", ["None"])[0] for x in data]).most_common()
common_categories

[('Pets & Animals', 4512),
 ('People & Blogs', 2877),
 ('Entertainment', 1412),
 ('Music', 771),
 ('Film & Animation', 567),
 ('Education', 497),
 ('None', 489),
 ('Comedy', 381),
 ('News & Politics', 296),
 ('Gaming', 239),
 ('Howto & Style', 121),
 ('Science & Technology', 97),
 ('Travel & Events', 57),
 ('Sports', 33),
 ('Autos & Vehicles', 26),
 ('Nonprofits & Activism', 15)]

In [7]:
chosen_categories = set([
    ('Pets & Animals', 4512), # almost sure there are cats on them
    ('Entertainment', 1412),  # The most noisy of all of them
    ('People & Blogs', 2877), # needs to be revised
    ('Education', 497),      # I saw toturials on how to recognise pain
    ('Howto & Style', 121),  # Has only 121 videos, worth checking
    ('Nonprofits & Activism', 15), # Maybw they are about rescuing cats?
    ('None', 489)            # These are the ones the crawled gathered before categories were added
])

In [8]:
chosen_categories = set([
    'Pets & Animals', # almost sure there are cats on them
    'Entertainment',  # The most noisy of all of them
    'People & Blogs', # needs to be revised
    'Education',       # I saw toturials on how to recognise pain
    'Howto & Style',  # Has only 121 videos, worth checking
    'Nonprofits & Activism',  # Maybw they are about rescuing cats?
    'None',         # These are the ones the crawled gathered before categories were added
])

In [9]:
sum([x[1] for x in chosen_categories]), len(data)

TypeError: unsupported operand type(s) for +: 'int' and 'str'

In [10]:
sum([x[1] for x in chosen_categories]) / len(data) * 100, 100

TypeError: unsupported operand type(s) for +: 'int' and 'str'

Filtered 20% of those videos already

In [11]:
tags_str = Counter([tag for x in data for tag in x.get("tags", [])])
tags_str

Counter({'Animal rescue': 11,
         'kitten rescue': 57,
         'cat': 2009,
         'cats': 1448,
         'kitten': 968,
         'kittens': 573,
         'cat rescue': 122,
         'rescue kitten': 41,
         'rescuepoorkittens': 2,
         'attempt': 2,
         'Cat tv for cats to watch': 1,
         'cat tv': 141,
         'nature escape': 3,
         'Montagem Miau': 1,
         'Ay mi gatito miau miau tiktok': 1,
         'Ay mi gatito tiktok': 1,
         'Miau miau tiktok': 1,
         'Miau tiktok': 1,
         'Ay mi gatito miau tiktok': 1,
         'Ay mi gatito miau miau': 1,
         'Ay mi gatito miau': 1,
         'ay mi gatito': 3,
         'ay mi gatito miau miau': 8,
         'miau miau': 10,
         'ay mi gatito tiktok': 1,
         'tiktok': 61,
         'miau miau tiktok': 2,
         'montagem miau funk': 1,
         'miau funk': 1,
         'montagem miau': 3,
         'montagem': 1,
         'miau miau funk': 1,
         'funk': 2,
         'Funk':

if we have ~10 000 videos, then it would be 1000 LLM calls with 10 videos in each

## Gemini Labeling

In [12]:
SYSTEM_PROMPT = """
### ROLE
You are an expert AI Research Assistant specializing in Feline Ethology and Computer Vision dataset curation. Your goal is to filter YouTube metadata to identify videos containing real, physical cats suitable for behavioral and emotional analysis.

### TASK
Evaluate a list of video metadata (titles, tags, descriptions) to determine if a real cat is the primary subject of the footage.

### INCLUSION & EXCLUSION LOGIC
1. **KEEP (True):**
   - Direct footage of domestic or wild cats (home videos, vet visits, grooming, rescues).
   - "Cat memes" or "Funny cat" compilations, provided they feature real animals.
   - Vocalization-heavy videos (cats meowing, hissing, or purring).
   - Multilingual content: Interpret meanings in any language (e.g., "Gato," "Кіт," "猫") as "Cat."

2. **EXCLUDE (False):**
   - **"Cat TV" / Entertainment for Cats:** Videos of birds, mice, or squirrels intended for cats to watch.
   - **Synthetic/Artistic:** CGI, 3D animations, cartoons (e.g., Simon's Cat), or plush/toy cats.
   - **Non-Primary:** Human-centric vlogs where a cat is mentioned but not shown, or "reaction" videos where the cat is a tiny thumbnail.
   - **Gaming:** Minecraft cats, Stray (the game), or virtual pets.

### UNCERTAINTY HEURISTIC
- Behavioral research requires a "wide net." If the metadata is vague but suggests a real cat *might* be present (e.g., a title like "My best friend" with a "cat" tag), **set "keep": true**.
- Only exclude if the metadata explicitly points to one of the "EXCLUDE" categories.

### OUTPUT FORMAT
Return **only** a valid JSON array of objects. Do not include conversational filler.
Each object must contain:
- "video_id": string
- "keep": boolean
- "reason": A 1-sentence explanation (e.g., "Metadata suggests real feline vocalization and physical presence.")
- "confidence": "high" (explicit mention), "medium" (likely real), or "low" (vague but kept).

Output format: a JSON array like this:
[
  {{
    "video_id": "abc123",
    "keep": true,
    "reason": "Title and tags indicate a real cat in the footage.",
    "confidence": "high"
  }},
  ...
]

### INPUT DATA
{videos_metadata_here}
"""

In [13]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

In [68]:
chosen_data = [x for x in data if x.get("categories", ["None"])[0] in chosen_categories]

BATCH = 10
MODEL = "google/gemini-2.5-flash"
OUTPUT_FILE = "gemini_labeled_videos.jsonl"

def clean_gemini_response(text):
    # Use regex to find the first '[' and the last ']'
    # This ignores any "Sure! Here is your JSON" text.
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if match:
        return match.group(0)
    return text.strip()

with open(OUTPUT_FILE, "w", encoding="utf-8") as f_out:

    for i in tqdm(range(0, len(chosen_data), BATCH), desc="Processing videos"):
        batch = chosen_data[i:i+BATCH]

        prompt = SYSTEM_PROMPT.format(videos_metadata_here=batch)

        completion = client.chat.completions.create(
            model=MODEL, 
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"} # <--- Add this line
        )


        response_text = completion.choices[0].message.content
        response_text = clean_gemini_response(response_text)

        try:
            # Parse the response text
            response_json = json.loads(response_text)
            
            # Robust extraction: handle both list and dict responses
            if isinstance(response_json, list):
                evaluations = response_json
            elif isinstance(response_json, dict):
                evaluations = response_json.get("evaluations", [])
            else:
                evaluations = []
            
            for video_obj in evaluations:
                f_out.write(json.dumps(video_obj, ensure_ascii=False) + "\n")
                
        except json.JSONDecodeError:
            print(f"Warning: Failed to parse JSON for batch {i}-{i+BATCH}")
            with open("error_log.txt", "a") as log:
                log.write(f"--- Batch {i} ---\n{response_text}\n\n")
            continue

        # Write each video object as a separate line in .jsonl
        for video_obj in response_json:
            f_out.write(json.dumps(video_obj, ensure_ascii=False) + "\n")

Processing videos:   1%|          | 6/993 [00:16<39:12,  2.38s/it]  

Processing videos:  10%|▉         | 98/993 [05:24<34:24,  2.31s/it]  

Processing videos:  14%|█▎        | 136/993 [07:34<51:05,  3.58s/it]

Processing videos:  40%|████      | 398/993 [22:06<25:01,  2.52s/it]  

Processing videos:  43%|████▎     | 429/993 [23:46<34:55,  3.71s/it]

Processing videos:  46%|████▌     | 458/993 [25:29<29:46,  3.34s/it]


KeyboardInterrupt: 

In [15]:
import json
import re
import os
from tqdm import tqdm

# --- CONFIGURATION ---
BATCH = 10
MODEL = "google/gemini-2.5-flash" 
OUTPUT_FILE = "gemini_labeled_videos.jsonl"

In [ ]:


def clean_gemini_response(text):
    """Extracts JSON structure from text."""
    match = re.search(r'(\{.*\}|\[.*\])', text, re.DOTALL)
    if match:
        return match.group(0)
    return text.strip()

# 1. PREPARE DATA
# Filter your initial data by category
chosen_data = [x for x in data if x.get("categories", ["None"])[0] in chosen_categories]

# 2. RESUME LOGIC: Check for existing progress
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f_check:
        for line in f_check:
            try:
                item = json.loads(line)
                processed_ids.add(item["video_id"])
            except:
                continue
    print(f"Resuming progress: {len(processed_ids)} videos already processed.")

# Filter out already processed videos
remaining_data = [v for v in chosen_data if v["video_id"] not in processed_ids]
print(f"Total to process: {len(remaining_data)}")

# 3. PROCESSING LOOP
# Open in 'a' (append) mode instead of 'w' (write)
with open(OUTPUT_FILE, "a", encoding="utf-8") as f_out:
    for i in tqdm(range(0, len(remaining_data), BATCH), desc="Processing videos"):
        batch = remaining_data[i : i + BATCH]
        
        prompt = SYSTEM_PROMPT.format(videos_metadata_here=batch)

        try:
            completion = client.chat.completions.create(
                model=MODEL, 
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"}
            )

            response_text = completion.choices[0].message.content
            cleaned_text = clean_gemini_response(response_text)
            response_json = json.loads(cleaned_text)
            
            # Handle both list and dict formats
            if isinstance(response_json, list):
                evaluations = response_json
            elif isinstance(response_json, dict):
                evaluations = response_json.get("evaluations", [])
            else:
                evaluations = []
            
            # Save progress
            for video_obj in evaluations:
                f_out.write(json.dumps(video_obj, ensure_ascii=False) + "\n")
            
            # Flush to disk frequently so progress is saved even if script crashes
            f_out.flush()

        except json.JSONDecodeError:
            print(f"Warning: Failed to parse JSON for batch starting at index {i}")
            with open("error_log.txt", "a") as log:
                log.write(f"--- Error at remaining_data index {i} ---\n{response_text}\n\n")
        except Exception as e:
            print(f"Request Error: {e}")
            continue

Resuming progress: 4530 videos already processed.
Total to process: 5401


Processing videos:   4%|▍         | 23/541 [01:12<26:41,  3.09s/it]

Processing videos:   9%|▉         | 51/541 [02:47<27:23,  3.35s/it]

Processing videos:  11%|█▏        | 62/541 [03:23<28:05,  3.52s/it]

Processing videos:  19%|█▉        | 102/541 [05:42<23:18,  3.18s/it]

...

Processing videos:  44%|████▎     | 236/541 [13:31<14:05,  2.77s/it]

Processing videos:  71%|███████   | 384/541 [22:10<07:56,  3.03s/it]

Processing videos:  75%|███████▍  | 405/541 [23:25<08:52,  3.91s/it]

Processing videos:  90%|█████████ | 488/541 [28:20<03:09,  3.57s/it]

Processing videos: 100%|██████████| 541/541 [31:19<00:00,  3.47s/it]


## Deduplication

In [16]:
ids = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f_check:
    for line in f_check:
        item = json.loads(line)
        ids.append(item)

In [17]:
len(ids)

14351

In [18]:
all_videos = set([id['video_id'] for id in ids])
len(all_videos)

9821

In [19]:
picked_videos = set([id['video_id'] for id in ids if id['keep']])
len(picked_videos)

8259

In [20]:
len(picked_videos) / len(all_videos) * 100

84.09530597698809

In [23]:
with open("logs/additional_videos_crawled.json", "w") as f:
    f.write(json.dumps([{"id": vid} for vid in picked_videos]))